# Audit Total Mounting Tiang SINERGI


## tl;dr

- Snapshot aktif memiliki **161 tiang** dan **660 perangkat yang dapat dimount**.
- Baru **297 perangkat (45.0%)** mempunyai relasi mounting; **363 perangkat** belum dimount.
- Dari perangkat yang belum dimount, **91** eksplisit berlabel indoor/non-tiang. Setelah dikeluarkan, coverage mounting menjadi **52.2%**.
- Hanya **3 tiang** yang benar-benar kosong. Masalah utama adalah perangkat yang belum terikat, bukan inventaris tiang yang hilang.
- **208 perangkat** belum dimount walaupun memiliki opsi tiang dalam 25 m; **155** tidak mempunyai opsi dalam radius tersebut.


## Context & Methods

Audit menggunakan proyeksi topology dataset aktif cabang Semarang dan merekonsiliasi inventaris aset, graph node, mountingRelations, mountingOptions, mountingCandidates, serta mountingOverrides. Grain utama adalah satu perangkat yang dapat dimount; grain tiang dihitung terpisah.

### Key Assumptions

- Tiang ditentukan dari klasifikasi pole/tiang/physical mount pada point graph node.
- Perangkat mountable mengikuti klasifikasi backend: kamera, junction box, dan JB rack yang cocok.
- Opsi 25 m bukan relasi terkonfirmasi; opsi hanya menunjukkan kandidat penetapan manual.


## Data


In [ ]:
from pathlib import Path
import json

results = json.loads(Path('mounting_audit_results.json').read_text(encoding='utf-8'))
results['reconciliation']


{
  "official_pole_count": 161,
  "audited_pole_count": 161,
  "official_mountable_count": 660,
  "audited_mountable_count": 660,
  "official_relation_count": 297,
  "audited_relation_count": 297,
  "counts_match": true
}


## Results

### Overall mounting profile


In [ ]:
results['overall']


{
  "dataset_version_id": "dv-6b012fa0-ec62-495d-9e84-ef7e4d3dd093",
  "dataset_status": "active",
  "activated_at": "2026-08-20T07:22:15.736Z",
  "mounting_generated_at": "2026-08-20T07:21:46.433Z",
  "asset_rows": 1376,
  "point_graph_nodes": 824,
  "poles": 161,
  "mountable_assets": 660,
  "mounted_assets": 297,
  "unmounted_assets": 363,
  "explicit_indoor_assets": 91,
  "explicit_indoor_mounted": 0,
  "explicit_indoor_unmounted": 91,
  "expected_pole_mount_assets_excluding_explicit_indoor": 569,
  "expected_pole_mounted_excluding_explicit_indoor": 297,
  "expected_pole_unmounted_excluding_explicit_indoor": 272,
  "expected_pole_unmounted_with_option_25m": 160,
  "expected_pole_unmounted_without_option_25m": 112,
  "adjusted_mount_rate_excluding_explicit_indoor": 0.522,
  "mount_rate": 0.45,
  "occupied_poles": 158,
  "empty_poles": 3,
  "pole_occupancy_rate": 0.9814,
  "mounting_relations": 297,
  "automatic_relations": 297,
  "manual_relations": 0,
  "mounting_candidates": 0,
  

### Coverage by area


In [ ]:
area_fields = ['area', 'poles', 'occupied_poles', 'empty_poles', 'mountable_assets', 'mounted_assets', 'unmounted_assets', 'explicit_indoor_unmounted', 'mount_rate', 'adjusted_mount_rate_excluding_explicit_indoor', 'unmounted_with_option_25m', 'unmounted_without_option_25m']
[{field: row[field] for field in area_fields} for row in results['per_area']]


area                   | poles | occupied_poles | empty_poles | mountable_assets | mounted_assets | unmounted_assets | explicit_indoor_unmounted | mount_rate | adjusted_mount_rate_excluding_explicit_indoor | unmounted_with_option_25m | unmounted_without_option_25m
-----------------------+-------+----------------+-------------+------------------+----------------+------------------+---------------------------+------------+-----------------------------------------------+---------------------------+-----------------------------
FT LOMANIS             | 28    | 28             | 0           | 112              | 42             | 70               | 20                        | 0.375      | 0.4565                                        | 48                        | 22                          
ITC LPG CILACAP        | 16    | 16             | 0           | 77               | 29             | 48               | 6                         | 0.3766     | 0.4085                                       

### Why devices remain unmounted


In [ ]:
results['unmounted_reason_buckets']


reason               | assets | share_of_unmounted
---------------------+--------+-------------------
Tidak ada opsi ≤25 m | 155    | 0.427             
Opsi >10–25 m        | 108    | 0.2975            
Opsi >5–10 m         | 100    | 0.2755            


### Referential-integrity checks


In [ ]:
{key: value['count'] for key, value in results['integrity'].items()}


{
  "duplicate_source_assets": 0,
  "orphan_sources": 0,
  "orphan_targets": 0,
  "non_mountable_sources": 0,
  "non_pole_targets": 0,
  "unconfirmed_relations": 0,
  "cross_area_relations": 0
}


## Takeaways

1. Kesenjangan terbesar berasal dari radius otomasi 5 m yang konservatif dan belum adanya keputusan manual.
2. Opsi mounting tersedia untuk sebagian besar kasus yang dekat, tetapi kandidat di luar 5 m tidak otomatis menjadi antrean keputusan.
3. Integritas referensial relasi yang sudah terbentuk bersih; risiko utama adalah kelengkapan dan kebenaran pasangan, bukan orphan atau duplikasi.
4. Ketidakcocokan nomor JB–tiang harus ditinjau sebagai flag audit, karena nomor yang berbeda belum tentu salah secara fisik tetapi bertentangan dengan konvensi yang diharapkan pengguna.
